<a href="https://colab.research.google.com/github/Qudus-O/Financial-News-Sentiment/blob/main/finance_sentiment_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QLoRA on a 1.7B Model for Financial News Sentiment

Third notebook in the series. Same data, splits (stratified split, seed 42) and test
set as the DistilBERT and LoRA notebooks. The base model is now
`HuggingFaceTB/SmolLM2-1.7B`, about 25 times larger, loaded in 4-bit and fine-tuned
with LoRA adapters (QLoRA).

**Question:** can a 1.7B model be fine-tuned on a free Colab GPU, and does it beat the
67M-parameter DistilBERT?

A 1.7B model fine-tuned with QLoRA reaches 0.877 macro-F1 versus 0.835 for the DistilBERT full fine-tune.

In [1]:
!pip uninstall -y -q torchao
!pip install -q transformers datasets accelerate scikit-learn matplotlib peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.5 MB/s eta 0:00:00


In [2]:

import re
import numpy as np
import torch, sklearn, transformers, datasets
from transformers import set_seed

set_seed(42)
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__, "| sklearn", sklearn.__version__)
print("GPU available:", torch.cuda.is_available())

torch 2.11.0+cu130 | transformers 5.18.0 | datasets 4.8.5 | sklearn 1.6.1
GPU available: True


In [3]:
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 2**30, 1), "GB")

Tesla T4 14.6 GB


Hardware check: the GPU name and memory are printed so the run can be reproduced.

In [4]:
from collections import Counter
from datasets import load_dataset

ds = load_dataset("zeroshot/twitter-financial-news-sentiment")
LABELS = {0: "Bearish", 1: "Bullish", 2: "Neutral"}

print(ds)

README.md:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

sent_train.csv:   0%|          | 0.00/859k [00:00<?, ?B/s]

sent_valid.csv:   0%|          | 0.00/217k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2388
    })
})


In [5]:
norm = lambda t: re.sub(r"https?://\S+", "", t).strip().lower()
train_texts = {norm(t) for t in ds["train"]["text"]}
dup = [t for t in ds["validation"]["text"] if norm(t) in train_texts]

print(len(dup), "of", len(ds["validation"]), "validation texts also appear in train")
print(dup[:3])
print(len(ds["train"]), "train rows,", len(train_texts), "unique after normalizing")

38 of 2388 validation texts also appear in train
['Fed Chairman Jerome Powell puts lawmakers on notice that fiscal policy may need to play a bigger role countering do… https://t.co/URFHNRkmfh', 'Exxon is cutting billions of dollars in investments to keep its financial footing during the worst oil-price rout i… https://t.co/MI5vCEkhlm', '3M is exploring a sale of its drug delivery systems unit, which could fetch about $1 billion https://t.co/ISlbwO3FCd']
9543 train rows, 9437 unique after normalizing


In [6]:
raw_train = ds["train"].class_encode_column("label")
parts = raw_train.train_test_split(test_size=0.1, seed=42, stratify_by_column="label")
fit_raw, dev_raw = parts["train"], parts["test"]
test_raw = ds["validation"]
print(len(fit_raw), len(dev_raw), len(test_raw))

Stringifying the column:   0%|          | 0/9543 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/9543 [00:00<?, ? examples/s]

8588 955 2388


## Tokenization
`HuggingFaceTB/SmolLM2-1.7B` tokenizer. This model reads left to right only, so
classification uses the output at the last real token, and padding goes on the right.
The model has no padding token of its own, so the end-of-text token is used. Length
statistics come from the fit set. `max_length=64` keeps training memory low.

In [7]:
from transformers import AutoTokenizer

MODEL = "HuggingFaceTB/SmolLM2-1.7B"
MAX_LEN = 64
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("pad:", repr(tokenizer.pad_token), tokenizer.pad_token_id,
      "| eos:", repr(tokenizer.eos_token), "| padding side:", tokenizer.padding_side)
print(tokenizer("Company cuts full-year guidance after weak demand")["input_ids"])

lengths = np.array([len(tokenizer(t)["input_ids"]) for t in fit_raw["text"]])
print("token length percentiles (50/90/99/max):", np.percentile(lengths, [50, 90, 99, 100]))
print("share truncated at", MAX_LEN, ":", round(float((lengths > MAX_LEN).mean()), 4))

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

fit_ds = fit_raw.map(tokenize, batched=True)
dev_ds = dev_raw.map(tokenize, batched=True)
test_ds = test_raw.map(tokenize, batched=True)
print(fit_ds)

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

...

pad: '<|endoftext|>' 0 | eos: '<|endoftext|>' | padding side: right
[3750, 10830, 10672, 2073, 29, 3610, 5783, 990, 4238, 3516]
token length percentiles (50/90/99/max): [ 25.  46.  58. 105.]
share truncated at 64 : 0.0033


Map:   0%|          | 0/8588 [00:00<?, ? examples/s]

Map:   0%|          | 0/955 [00:00<?, ? examples/s]

Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 8588
})


In [8]:
print(MODEL)
print(type(tokenizer).__name__)
print(tokenizer.name_or_path)

HuggingFaceTB/SmolLM2-1.7B
GPT2Tokenizer
HuggingFaceTB/SmolLM2-1.7B


**What this shows:** SmolLM2's tokenizer adds no start or end tokens. It uses
`<|endoftext|>` (ID 0) for both end-of-text and padding, with padding on the right.
Median tweet length is 25 tokens, the 99th percentile is 58, and the longest is 105.
With `max_length=64`, 0.33% of fit tweets (about 28 of 8,588) are truncated.

##Loading the model in 4-bit
The frozen base model is stored in 4-bit numbers (NF4) to cut memory. During each
calculation the weights are expanded to 16-bit, and fp16 is used because the T4 has no
hardware bfloat16 support. The classification head is new and starts from random values.

In [9]:
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

torch.cuda.empty_cache()
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=3, quantization_config=bnb, device_map={"": 0})
model.config.pad_token_id = tokenizer.pad_token_id

print("GPU memory after loading (GB):", round(torch.cuda.memory_allocated() / 2**30, 2))
print("parameters (billions):", round(sum(p.numel() for p in model.parameters()) / 1e9, 2))

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

[transformers] LlamaForSequenceClassification LOAD REPORT from: HuggingFaceTB/SmolLM2-1.7B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


GPU memory after loading (GB): 0.96
parameters (billions): 0.91


## Preparing the 4-bit model and adding LoRA
`prepare_model_for_kbit_training` makes the quantized model safe to train. It converts
some layers to higher precision for stability and turns on gradient checkpointing,
which saves memory by recomputing activations during the backward pass instead of
storing them, at the cost of some speed. LoRA adapters (r=16) are then added to the four
attention projections. The new classification head is trained in full.

In [10]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

print("hidden size, layers, kv heads, attention heads:",
      model.config.hidden_size, model.config.num_hidden_layers,
      model.config.num_key_value_heads, model.config.num_attention_heads)

print("before prep (GiB):", round(torch.cuda.memory_allocated() / 2**30, 2))
model = prepare_model_for_kbit_training(model)
print("after prep (GiB): ", round(torch.cuda.memory_allocated() / 2**30, 2))

cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    modules_to_save=["score"],
)
model = get_peft_model(model, cfg)
model.print_trainable_parameters()

hidden size, layers, kv heads, attention heads: 2048 24 32 32
before prep (GiB): 0.96
after prep (GiB):  1.15
trainable params: 6,297,600 || all params: 1,717,680,128 || trainable%: 0.3666


**What this shows:** 6,297,600 trainable parameters (0.37% of 1.72B). Of these,
6,291,456 are LoRA matrices (r=16 on four 2048x2048 projections in each of 24 layers)
and 6,144 are the classification head. The model has 32 key/value heads and 32
attention heads, so all four projections are square. Preparing the model for training
raised GPU memory from 0.96 to 1.15 GiB, which is consistent with converting the
roughly 100M-parameter embedding table to 32-bit.

## Trial run
Forty optimizer steps to check that training works and to measure speed and memory
before the real run. This run changes the adapter weights, so the real runs rebuild the
model from scratch.

In [11]:
import time
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer, set_seed
from sklearn.metrics import accuracy_score, f1_score

collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro")}

set_seed(0)
args = TrainingArguments(
    output_dir="qlora-trial", max_steps=40,
    per_device_train_batch_size=16, gradient_accumulation_steps=2,
    learning_rate=2e-4, warmup_steps=5, logging_steps=10,
    eval_strategy="no", save_strategy="no", fp16=True,
    report_to="none", seed=0, label_names=["labels"], disable_tqdm=True)

trainer = Trainer(model=model, args=args, train_dataset=fit_ds,
                  data_collator=collator, processing_class=tokenizer)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer.train()
secs = time.time() - t0

steps_per_epoch = len(fit_ds) / (16 * 2)
print("seconds:", round(secs, 1), "| per optimizer step:", round(secs / 40, 2), "s")
print("peak GPU memory (GiB):", round(torch.cuda.max_memory_allocated() / 2**30, 2))
print("optimizer steps per epoch:", round(steps_per_epoch),
      "| projected minutes per epoch:", round(steps_per_epoch * secs / 40 / 60, 1))

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': '2.486', 'grad_norm': '15.23', 'learning_rate': '0.0001771', 'epoch': '0.03724'}
{'loss': '1.827', 'grad_norm': '18.69', 'learning_rate': '0.00012', 'epoch': '0.07449'}
{'loss': '1.616', 'grad_norm': '13.45', 'learning_rate': '6.286e-05', 'epoch': '0.1117'}
{'loss': '1.894', 'grad_norm': '9.485', 'learning_rate': '5.714e-06', 'epoch': '0.149'}
{'train_runtime': '62.91', 'train_samples_per_second': '20.34', 'train_steps_per_second': '0.636', 'train_loss': '1.956', 'epoch': '0.149'}
seconds: 63.8 | per optimizer step: 1.59 s
peak GPU memory (GiB): 1.7
optimizer steps per epoch: 268 | projected minutes per epoch: 7.1


## One full epoch
A full epoch (268 optimizer steps) with the loss logged every 25 steps and dev scored
at the end. Check: the loss should fall below about 0.89 (the loss of a model that only
knows the class proportions) and dev macro-F1 should be well above 0.26 (the
majority-class level). The model is rebuilt from scratch, so the trial run does not
carry over.

In [12]:
import gc

for name in ["trainer", "model"]:
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

def build_qlora(seed=0, r=16):
    set_seed(seed)
    m = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=3, quantization_config=bnb, device_map={"": 0})
    m.config.pad_token_id = tokenizer.pad_token_id
    m = prepare_model_for_kbit_training(m)
    c = LoraConfig(task_type=TaskType.SEQ_CLS, r=r, lora_alpha=2 * r, lora_dropout=0.05,
                   target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                   modules_to_save=["score"])
    return get_peft_model(m, c)

model = build_qlora(seed=0)
args = TrainingArguments(
    output_dir="qlora-1ep", num_train_epochs=1,
    per_device_train_batch_size=16, gradient_accumulation_steps=2,
    per_device_eval_batch_size=32,
    learning_rate=2e-4, warmup_steps=13, lr_scheduler_type="linear",
    logging_steps=25, eval_strategy="epoch", save_strategy="no",
    fp16=True, report_to="none", seed=0, label_names=["labels"], disable_tqdm=True)
trainer = Trainer(model=model, args=args, train_dataset=fit_ds, eval_dataset=dev_ds,
                  data_collator=collator, processing_class=tokenizer,
                  compute_metrics=compute_metrics)

t0 = time.time()
trainer.train()
print("minutes:", round((time.time() - t0) / 60, 1))

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

[transformers] LlamaForSequenceClassification LOAD REPORT from: HuggingFaceTB/SmolLM2-1.7B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': '2.104', 'grad_norm': '33.8', 'learning_rate': '0.0001914', 'epoch': '0.09311'}
{'loss': '1.665', 'grad_norm': '9.198', 'learning_rate': '0.0001719', 'epoch': '0.1862'}
{'loss': '1.456', 'grad_norm': '17.26', 'learning_rate': '0.0001523', 'epoch': '0.2793'}
{'loss': '1.213', 'grad_norm': '17.16', 'learning_rate': '0.0001328', 'epoch': '0.3724'}
{'loss': '1.032', 'grad_norm': '13.63', 'learning_rate': '0.0001133', 'epoch': '0.4655'}
{'loss': '0.7334', 'grad_norm': '10.86', 'learning_rate': '9.375e-05', 'epoch': '0.5587'}
{'loss': '0.7436', 'grad_norm': '12.64', 'learning_rate': '7.422e-05', 'epoch': '0.6518'}
{'loss': '0.7235', 'grad_norm': '14.63', 'learning_rate': '5.469e-05', 'epoch': '0.7449'}
{'loss': '0.734', 'grad_norm': '12.4', 'learning_rate': '3.516e-05', 'epoch': '0.838'}
{'loss': '0.6285', 'grad_norm': '13.97', 'learning_rate': '1.563e-05', 'epoch': '0.9311'}
{'eval_loss': '0.3422', 'eval_accuracy': '0.8691', 'eval_macro_f1': '0.8281', 'eval_runtime': '14.15', 'eval

**What this shows (1 epoch, seed 0, dev set):** dev accuracy 0.869, macro-F1 0.828,
eval loss 0.342. The 40-step trial had a loss above 1.1, so the setup trains. The epoch
took 7.3 minutes. The dev set has 955 rows and this is one seed, so differences under
about 0.01 against other models are not meaningful.

## QLoRA final runs
Settings were fixed in advance and not tuned on the test set: r=16, learning rate 2e-4,
3 epochs, effective batch size 32 (16 x 2 accumulation), 4-bit NF4 base model, LoRA on
the four attention projections. The 3-epoch choice follows the DistilBERT protocol, and
a one-epoch dev check confirmed that training works. Dev macro-F1 is logged after every
epoch. The test set is scored once per seed. Seeds 0, 1 and 2, each in its own cell.

In [13]:
import os, shutil

def final_qlora(seed, epochs=3, lr=2e-4, r=16):
    for name in ["trainer", "model"]:
        globals().pop(name, None)
    gc.collect(); torch.cuda.empty_cache()

    m = build_qlora(seed=seed, r=r)
    args = TrainingArguments(
        output_dir=f"qlora-final-{seed}", num_train_epochs=epochs,
        per_device_train_batch_size=16, gradient_accumulation_steps=2,
        per_device_eval_batch_size=32, learning_rate=lr,
        warmup_steps=13, lr_scheduler_type="linear", logging_strategy="no",
        eval_strategy="epoch", save_strategy="no", fp16=True,
        report_to="none", seed=seed, label_names=["labels"], disable_tqdm=True)
    t = Trainer(model=m, args=args, train_dataset=fit_ds, eval_dataset=dev_ds,
                data_collator=collator, processing_class=tokenizer,
                compute_metrics=compute_metrics)
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    t.train()
    secs = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 2**30
    curve = [round(h["eval_macro_f1"], 4) for h in t.state.log_history
             if "eval_macro_f1" in h]
    out = t.predict(test_ds)
    path = f"qlora-adapter-{seed}"
    shutil.rmtree(path, ignore_errors=True)
    m.save_pretrained(path)
    size = sum(os.path.getsize(os.path.join(path, f)) for f in os.listdir(path)) / 2**20
    return {"seed": seed, "minutes": round(secs / 60, 1), "peak_GiB": round(peak, 2),
            "adapter_MB": round(size, 1), "dev_curve": curve,
            "test_acc": out.metrics["test_accuracy"],
            "test_macro_f1": out.metrics["test_macro_f1"],
            "preds": np.argmax(out.predictions, axis=-1)}

qlora_runs = [final_qlora(0)]
print({k: v for k, v in qlora_runs[0].items() if k != "preds"})

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

[transformers] LlamaForSequenceClassification LOAD REPORT from: HuggingFaceTB/SmolLM2-1.7B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'eval_loss': '0.3154', 'eval_accuracy': '0.8806', 'eval_macro_f1': '0.8425', 'eval_runtime': '14.11', 'eval_samples_per_second': '67.67', 'eval_steps_per_second': '2.126', 'epoch': '1'}


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'eval_loss': '0.2736', 'eval_accuracy': '0.9005', 'eval_macro_f1': '0.8665', 'eval_runtime': '14.22', 'eval_samples_per_second': '67.15', 'eval_steps_per_second': '2.109', 'epoch': '2'}


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'eval_loss': '0.2891', 'eval_accuracy': '0.8974', 'eval_macro_f1': '0.8653', 'eval_runtime': '14.18', 'eval_samples_per_second': '67.36', 'eval_steps_per_second': '2.116', 'epoch': '3'}
{'train_runtime': '1348', 'train_samples_per_second': '19.11', 'train_steps_per_second': '0.599', 'train_loss': '0.6097', 'epoch': '3'}
{'seed': 0, 'minutes': 22.5, 'peak_GiB': 1.7, 'adapter_MB': 24.1, 'dev_curve': [0.8425, 0.8665, 0.8653], 'test_acc': 0.9053601340033501, 'test_macro_f1': 0.8765990471122308}


**What this shows (seed 0 only, test set):** QLoRA on SmolLM2-1.7B reaches 0.905
accuracy and 0.877 macro-F1, above the DistilBERT full fine-tune (0.874 and 0.835, mean
of 3 seeds). Dev macro-F1 per epoch was 0.843, 0.867, 0.865, flat after epoch 2. Settings
were fixed before looking at test. Training took 22 minutes, with 6.3M trainable
parameters and a 24 MB adapter, which needs the base model to run. Dev scoring ran at
about 72 rows/s, roughly 30 times slower than DistilBERT (rough figure: different batch
sizes, GPU not checked). This is one seed, so the gap is not yet confirmed.

In [14]:
qlora_runs.append(final_qlora(1))
print({k: v for k, v in qlora_runs[-1].items() if k != "preds"})

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

[transformers] LlamaForSequenceClassification LOAD REPORT from: HuggingFaceTB/SmolLM2-1.7B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'eval_loss': '0.3149', 'eval_accuracy': '0.8712', 'eval_macro_f1': '0.826', 'eval_runtime': '14.12', 'eval_samples_per_second': '67.62', 'eval_steps_per_second': '2.124', 'epoch': '1'}


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'eval_loss': '0.3047', 'eval_accuracy': '0.889', 'eval_macro_f1': '0.844', 'eval_runtime': '14.2', 'eval_samples_per_second': '67.27', 'eval_steps_per_second': '2.113', 'epoch': '2'}


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'eval_loss': '0.2929', 'eval_accuracy': '0.9026', 'eval_macro_f1': '0.8721', 'eval_runtime': '14.12', 'eval_samples_per_second': '67.64', 'eval_steps_per_second': '2.125', 'epoch': '3'}
{'train_runtime': '1358', 'train_samples_per_second': '18.98', 'train_steps_per_second': '0.594', 'train_loss': '0.626', 'epoch': '3'}
{'seed': 1, 'minutes': 22.6, 'peak_GiB': 1.7, 'adapter_MB': 24.1, 'dev_curve': [0.826, 0.844, 0.8721], 'test_acc': 0.9041038525963149, 'test_macro_f1': 0.8753925290723749}


**What this shows (test set):** Seeds 0 and 1 give 0.905 and 0.904 accuracy and 0.877 and
0.875 macro-F1. Dev curves differ more between seeds (epoch-3 dev macro-F1 0.865 vs
0.872, with a different shape across epochs) than test scores do, which is consistent
with a 955-row dev set being too noisy to rank settings. A PyTorch warning about
`use_reentrant` appears during training and is harmless.

In [15]:
qlora_runs.append(final_qlora(2))
print({k: v for k, v in qlora_runs[-1].items() if k != "preds"})

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

[transformers] LlamaForSequenceClassification LOAD REPORT from: HuggingFaceTB/SmolLM2-1.7B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'eval_loss': '0.2945', 'eval_accuracy': '0.8869', 'eval_macro_f1': '0.847', 'eval_runtime': '14.19', 'eval_samples_per_second': '67.32', 'eval_steps_per_second': '2.115', 'epoch': '1'}


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'eval_loss': '0.2637', 'eval_accuracy': '0.9005', 'eval_macro_f1': '0.8723', 'eval_runtime': '14.23', 'eval_samples_per_second': '67.09', 'eval_steps_per_second': '2.107', 'epoch': '2'}


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'eval_loss': '0.2705', 'eval_accuracy': '0.9047', 'eval_macro_f1': '0.8778', 'eval_runtime': '14.17', 'eval_samples_per_second': '67.38', 'eval_steps_per_second': '2.117', 'epoch': '3'}
{'train_runtime': '1359', 'train_samples_per_second': '18.96', 'train_steps_per_second': '0.594', 'train_loss': '0.6', 'epoch': '3'}
{'seed': 2, 'minutes': 22.7, 'peak_GiB': 1.7, 'adapter_MB': 24.1, 'dev_curve': [0.847, 0.8723, 0.8778], 'test_acc': 0.9066164154103853, 'test_macro_f1': 0.8802878486860699}


**What this shows (test set, 3 seeds):** QLoRA on SmolLM2-1.7B reaches 0.905 accuracy and
0.877 macro-F1 (seed std about 0.002), against 0.874 and 0.835 for the DistilBERT full
fine-tune. The gap of 0.042 is about twenty times the seed spread, and every QLoRA seed
beats every DistilBERT seed. The comparison changes both the method and the model
(1.7B decoder vs 67M encoder), so the gain should not be attributed to QLoRA alone:
QLoRA is what made the larger model trainable on a free 14.6 GB GPU. Each run took about
22 minutes, peaked at 1.7 GiB of GPU memory, and saved a 24 MB adapter that needs the
base model to run.

In [17]:
from sklearn.metrics import precision_recall_fscore_support
import pandas as pd

q_df = pd.DataFrame([{k: v for k, v in r_.items() if k not in ("preds", "dev_curve")}
                     for r_ in qlora_runs])
print(q_df.round(4).to_string(index=False))
print("test acc      mean", round(q_df.test_acc.mean(), 4), "std", round(q_df.test_acc.std(ddof=0), 4))
print("test macro-F1 mean", round(q_df.test_macro_f1.mean(), 4), "std", round(q_df.test_macro_f1.std(ddof=0), 4))

y = np.array(test_raw["label"])
per = np.array([
    np.stack(precision_recall_fscore_support(y, r_["preds"], labels=[0, 1, 2], zero_division=0)[:3])
    for r_ in qlora_runs])
tbl = pd.DataFrame({
    "class": ["Bearish", "Bullish", "Neutral"],
    "precision": per[:, 0].mean(0), "recall": per[:, 1].mean(0),
    "f1": per[:, 2].mean(0), "recall_std": per[:, 1].std(0)})
print(tbl.round(3).to_string(index=False))

 seed  minutes  peak_GiB  adapter_MB  test_acc  test_macro_f1
    0     22.5       1.7        24.1    0.9054         0.8766
    1     22.6       1.7        24.1    0.9041         0.8754
    2     22.7       1.7        24.1    0.9066         0.8803
test acc      mean 0.9054 std 0.001
test macro-F1 mean 0.8774 std 0.0021
  class  precision  recall    f1  recall_std
Bearish      0.836   0.838 0.837       0.013
Bullish      0.866   0.858 0.862       0.004
Neutral      0.932   0.935 0.934       0.002


## Results by class
Test set, mean of 3 seeds. TF-IDF and DistilBERT columns come from the first notebook
(same data, splits and test set).

| Class | Metric | TF-IDF | DistilBERT full | QLoRA 1.7B |
|---|---|---|---|---|
| Bearish | precision | 0.797 | 0.767 | 0.836 |
| | recall | 0.406 | 0.804 | 0.838 |
| | F1 | 0.538 | 0.785 | 0.837 |
| Bullish | precision | 0.796 | 0.816 | 0.866 |
| | recall | 0.558 | 0.796 | 0.858 |
| | F1 | 0.656 | 0.806 | 0.862 |
| Neutral | precision | 0.804 | 0.916 | 0.932 |
| | recall | 0.964 | 0.913 | 0.935 |
| | F1 | 0.877 | 0.914 | 0.934 |

**What this shows:** QLoRA improves F1 over the DistilBERT full fine-tune by 0.056 on
Bullish, 0.052 on Bearish and 0.020 on Neutral. The Bearish gain comes mostly from
precision (+0.069), with recall up 0.034 (seed std 0.013). About 6.5% of Neutral tweets
are flagged as Bearish or Bullish, against 8.7% for DistilBERT and 3.6% for TF-IDF.

## Conclusions
- A 1.7B-parameter model fine-tuned with QLoRA on a free 14.6 GB Colab T4 reaches 0.877
  macro-F1 (0.905 accuracy) over 3 seeds, against 0.835 for a fully fine-tuned DistilBERT
  and 0.690 for TF-IDF.
- The gain is largest on the minority classes. About 84 of 100 bearish tweets are caught
  (80 for DistilBERT, 41 for TF-IDF).
- The comparison changes both the method and the model (1.7B decoder vs 67M encoder),
  so the gain should not be attributed to QLoRA alone. QLoRA is what made the larger
  model trainable on this GPU.
- Cost: about 22.5 minutes per run (DistilBERT full fine-tune: about 1 minute), 1.7 GiB
  peak GPU memory, 6.3M trainable parameters (0.37%), and a 24 MB adapter that needs the
  3.4 GB base model to run. Scoring was roughly 30 times slower than DistilBERT
  (rough figure: different batch sizes, GPU not checked).

**Limitations:** one dataset of short English financial tweets; three seeds per method;
QLoRA settings were fixed in advance and not tuned, and DistilBERT used default settings;
one larger model family was tested; the label ambiguity seen in the first notebook applies
here too.